In [ ]:
import time
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
from matplotlib.colors import Normalize
from PIL import Image

from pc import PS
from modules import ADC,DAC,CHIP
from command import CMD,CmdData,Packet
from command.singleCmdInfo import *

from util import plot_v_cond,plot_cond,show_crossbar

In [ ]:
chip=CHIP(PS(host="192.168.1.10", port = 7, debug=0),init=True)
chip.set_device_cfg(deviceType=0)

In [ ]:
fig_num = 0

In [ ]:
def set_cycle(times,start_tg_v = 1,delta_tg_v = 0.1, write_voltage = 3,lower_bound=600,pulse_width = 100e-6,show = False):
    voltage_base = np.zeros((256,256))
    voltage = np.zeros((256,256))
    need_read = np.ones((256,256),dtype=bool)
    for i in range(times):
        tgv = start_tg_v+i*delta_tg_v
        voltage_base[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)[need_read]
        voltage[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)[need_read]
        cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)
        condition_set = (cond_sub_base < lower_bound) & need_read
        need_read = condition_set

        global fig_num
        path = f"./temp/{fig_num}.png"
        plot_cond(cond_sub_base,vmax=1000,title=f"set_tgv={int(tgv*10)},need_set={int(np.sum(condition_set))}",path=path)
        fig_num +=1

        # set操作
        chip.write_point2(crossbar=condition_set,write_voltage=write_voltage,tg=tgv,pulse_width=pulse_width,set_device=True)


    if show:
        need_read = np.ones((256,256),dtype=bool)
        voltage_base = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
        voltage = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
        cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)

        plot_cond(cond_sub_base,title=f"read_after_set",vmax=1000)

def reset_cycle(times,start_v = 1,delta_v = 0.1,upper_bound=200,pulse_width = 100e-6,show = False):
    voltage_base = np.zeros((256,256))
    voltage = np.zeros((256,256))
    need_read = np.ones((256,256),dtype=bool)
    for i in range(times):
        reset_v = start_v+i*delta_v
        voltage_base[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)[need_read]
        voltage[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)[need_read]
        cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)
        condition_reset = (cond_sub_base > upper_bound) & need_read
        need_read = condition_reset

        global fig_num
        path = f"./temp/{fig_num}.png"
        plot_cond(cond_sub_base,vmax=1000,title=f"reset_v={int(reset_v*10)},need_reset={int(np.sum(condition_reset))}",path=path)
        fig_num +=1

        # reset操作
        chip.write_point2(crossbar=condition_reset,write_voltage=reset_v,tg=5,pulse_width=pulse_width,set_device=False)

    if show:
        need_read = np.ones((256,256),dtype=bool)
        voltage_base = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
        voltage = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
        cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)

        plot_cond(cond_sub_base,title=f"read_after_reset",vmax=1000)
        

In [ ]:
# set_cycle(1,start_tg_v=1.6,delta_tg_v=0.1,write_voltage=3, lower_bound=800,pulse_width=100e-6)
reset_cycle(2,start_v=3,delta_v = 0.1,upper_bound=200,pulse_width=10e-6)

In [ ]:
show_crossbar(chip,vmax=1000)

In [ ]:
cond_target = np.load("./result/cond_target.npy")
cond_sub_base = np.load("./result/cond_sub_base.npy")
plot_cond(cond_sub_base,vmax=1000)
voltage_base = np.zeros((256,256))
voltage = np.zeros((256,256))
need_read = (cond_sub_base-cond_target)>500
voltage_base[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)[need_read]
voltage[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)[need_read]
cond_sub_base[need_read] = chip.voltage_to_cond(voltage-voltage_base)[need_read]

np.save("./result/cond_sub_base2.npy",cond_sub_base)
plot_cond(cond_sub_base,vmax=1000)

In [ ]:
plot_cond(cond_sub_base,vmax=1000)

In [ ]:
# need_read = np.ones((256,256))#np.load("reset_point.npy")
# # 读器件
# voltage_base = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
# voltage = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
# cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)
# cond_sub_base_hist = cond_sub_base

In [ ]:
voltage_base = np.zeros((256,256))
voltage = np.zeros((256,256))
need_read = (cond_sub_base-cond_target)>500
for i in range(6):
    reset_v = 2.5+i*0.1
    voltage_base[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)[need_read]
    voltage[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)[need_read]
    cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)
    condition_reset = ((cond_sub_base-cond_target)> 200) & need_read
    need_read = condition_reset

    interval = 20
    bin_edges = np.linspace(5, 2000, int(2000/interval)+1)    
    data = cond_sub_base.flatten()
    counts, bin_edges, _ = plt.hist(data, bins=bin_edges, color='blue', alpha=0.7, edgecolor='black')
    max_count = np.max(counts)
    max_index = np.argmax(counts)
    print(max_index,max_count)
    # 添加标题和标签
    plt.title(f"{int(reset_v*10)}v_interval{interval}_{int(max_index)*interval}us_{int(max_index+1)*interval}us")
    plt.xlabel("cond(uS)")
    plt.ylabel("Frequency")
    plt.show()

    chip.write_point2(crossbar=condition_reset,write_voltage=reset_v,tg=5,pulse_width=1e-6,set_device=False)

In [ ]:
voltage_base = np.zeros((256,256))
voltage = np.zeros((256,256))
need_read = (cond_sub_base-cond_target)>300
for i in range(11):
    tgv = 2.5+i*0.1
    voltage_base[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)[need_read]
    voltage[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)[need_read]
    cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)
    condition_set = ((cond_sub_base-cond_target) >200) & need_read

    need_read = condition_set

    interval = 20
    bin_edges = np.linspace(5, 2000, int(2000/interval)+1)    
    data = cond_sub_base.flatten()
    counts, bin_edges, _ = plt.hist(data, bins=bin_edges, color='blue', alpha=0.7, edgecolor='black')
    max_count = np.max(counts)
    max_index = np.argmax(counts)
    print(max_index,max_count)
    # 添加标题和标签
    plt.title(f"{int(tgv*10)}v_interval{interval}_{int(max_index)*interval}us_{int(max_index+1)*interval}us")
    plt.xlabel("cond(uS)")
    plt.ylabel("Frequency")
    plt.show()

    # set操作
    chip.write_point2(crossbar=condition_set,write_voltage=tgv,tg=5,pulse_width=100e-6,set_device=True)


# need_read = np.ones((256,256),dtype=bool)
# voltage_base = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
# voltage = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
# cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)

# plot_cond(cond_sub_base,title=f"read_after_set",vmax=1000)

In [ ]:
voltage_base = np.zeros((256,256))
voltage = np.zeros((256,256))
need_read = np.ones((256,256),dtype=bool)
for i in range(21):
    tgv = 2+i*0.05
    voltage_base[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)[need_read]
    voltage[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)[need_read]
    cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)
    condition_set = (cond_sub_base >800) & need_read
    need_read = condition_set

    global fig_num
    path = f"./temp/{fig_num}.png"
    plot_cond(cond_sub_base,vmax=1000,title=f"set_tgv={int(tgv*10)},need_set={int(np.sum(condition_set))}",path=path)
    fig_num +=1

    # set操作
    chip.write_point2(crossbar=condition_set,write_voltage=tgv,tg=5,pulse_width=1000e-6,set_device=True)


need_read = np.ones((256,256),dtype=bool)
voltage_base = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
voltage = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)

plot_cond(cond_sub_base,title=f"read_after_set",vmax=1000)

In [ ]:
voltage_base = np.zeros((256,256))
voltage = np.zeros((256,256))
need_read = np.ones((256,256),dtype=bool)
for i in range(41):
    tgv = 2+i*0.05
    voltage_base[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)[need_read]
    voltage[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)[need_read]
    cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)
    condition_set = (cond_sub_base >800)& (cond_sub_base <1400) & need_read
    need_read = condition_set

    global fig_num
    path = f"./temp/{fig_num}.png"
    plot_cond(cond_sub_base,vmax=1000,title=f"set_tgv={int(tgv*10)},need_set={int(np.sum(condition_set))}",path=path)
    fig_num +=1

    # set操作
    chip.write_point2(crossbar=condition_set,write_voltage=tgv,tg=5,pulse_width=1000e-6,set_device=True)


need_read = np.ones((256,256),dtype=bool)
voltage_base = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
voltage = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)

plot_cond(cond_sub_base,title=f"read_after_set",vmax=1000)

In [ ]:
# voltage_base = np.zeros((256,256))
# voltage = np.zeros((256,256))
# need_read = np.ones((256,256),dtype=bool)
# for i in range(41):
#     tgv = 2+i*0.05
#     voltage_base[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)[need_read]
#     voltage[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)[need_read]
#     cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)
#     condition_set = (cond_sub_base >600)& (cond_sub_base <1300) & need_read
#     need_read = condition_set

#     global fig_num
#     path = f"./temp/{fig_num}.png"
#     plot_cond(cond_sub_base,vmax=1000,title=f"set_tgv={int(tgv*10)},need_set={int(np.sum(condition_set))}",path=path)
#     fig_num +=1

#     # set操作
#     chip.write_point2(crossbar=condition_set,write_voltage=tgv,tg=5,pulse_width=1000e-6,set_device=True)


# need_read = np.ones((256,256),dtype=bool)
# voltage_base = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
# voltage = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
# cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)

# plot_cond(cond_sub_base,title=f"read_after_set",vmax=1000)

In [ ]:
for i in range(10):
    set_cycle(26,start_tg_v=0.5,delta_tg_v=0.1,write_voltage=5, lower_bound=800,pulse_width=1e-3)

In [ ]:
voltage_base = np.zeros((256,256))
voltage = np.zeros((256,256))
need_read = np.ones((256,256),dtype=bool)
for i in range(21):
    tgv = 3
    voltage_base[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)[need_read]
    voltage[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)[need_read]
    cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)
    condition_set = (cond_sub_base >200) & need_read
    need_read = condition_set

    global fig_num
    path = f"./temp/{fig_num}.png"
    plot_cond(cond_sub_base,vmax=1000,title=f"set_tgv={int(tgv*10)},need_set={int(np.sum(condition_set))}",path=path)
    fig_num +=1

    # set操作
    chip.write_point2(crossbar=condition_set,write_voltage=tgv,tg=5,pulse_width=100e-6,set_device=True)


need_read = np.ones((256,256),dtype=bool)
voltage_base = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
voltage = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)

plot_cond(cond_sub_base,title=f"read_after_set",vmax=1000)

In [ ]:
reset_cycle(100,5,0,upper_bound=200,pulse_widtch=0.05)

In [ ]:
set_cycle(26,start_tg_v=0.5,delta_tg_v=0.1,lower_bound=800)

In [ ]:
set_cycle(46,start_tg_v=0.5,delta_tg_v=0.1,write_voltage=4, lower_bound=800)

In [ ]:
for v in [4,4.2,4.4,4.6,4.8,5]:
    set_cycle(46,start_tg_v=0.5,delta_tg_v=0.1,write_voltage=v, lower_bound=800)

In [ ]:
set_cycle(46,start_tg_v=0.5,delta_tg_v=0.1,write_voltage=5, lower_bound=800,pulse_widtch=0.01)

In [ ]:
set_cycle(46,start_tg_v=0.5,delta_tg_v=0.1,write_voltage=5, lower_bound=800,pulse_widtch=0.05)

In [ ]:
set_cycle(31,start_tg_v=3,delta_tg_v=0,write_voltage=5, lower_bound=800,pulse_widtch=0.05)

In [ ]:
# chip.ps.set_time_out(time_out=30)

In [ ]:
set_cycle(100,start_tg_v=3,delta_tg_v=0,write_voltage=5, lower_bound=600,pulse_widtch=0.01)

# 4. 判断tg和电导的映射关系

In [ ]:
interval = 20
bin_edges = np.linspace(0, 2000, (2000/interval)+1)

need_read = np.ones((256,256),dtype=bool)

for i in range(18):
    tg = 0.8+0.1*i
    # 先reset
    chip.write_point2(crossbar=need_read,write_voltage=2,tg=5,pulse_width=1e-3,set_device=False)
    # 再set
    chip.write_point2(crossbar=need_read,write_voltage=3,tg=tg,pulse_width=1e-3,set_device=True)
    # 读器件
    voltage_base = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
    voltage = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
    cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)

    cond = cond_sub_base.flatten()
    counts, bin_edges, _ = plt.hist(cond, bins=bin_edges, color='blue', alpha=0.7, edgecolor='black')
    np.save(f"result/tgv2/{int(tg*10)}v_interval{interval}_vbin_edges.npy", bin_edges)
    np.save(f"result/tgv2/{int(tg*10)}v_interval{interval}_bin_counts.npy", counts)
    max_count = np.max(counts)
    max_index = np.argmax(counts)
    print(max_index,max_count)
    # 添加标题和标签
    plt.title(f"{int(tg*10)}v_interval{interval}_{int(max_index)*interval}us_{int(max_index+1)*interval}us")
    plt.xlabel("cond(uS)")
    plt.ylabel("Frequency")

    plt.savefig(f"result/tgv2/{int(tg*10)}v_interval{interval}_{int(max_index)*interval}us_{int(max_index+1)*interval}us.png")  # 保存为 PNG 格式
    plt.show()

In [ ]:
interval = 20
bin_edges = np.linspace(0, 2000, int(2000/interval)+1)

need_read = np.ones((256,256),dtype=bool)

for i in range(16):
    tg = 0.8+0.1*i
    # 先reset
    chip.write_point2(crossbar=need_read,write_voltage=2,tg=5,pulse_width=1e-3,set_device=False)
    # 再set
    chip.write_point2(crossbar=need_read,write_voltage=3,tg=tg,pulse_width=1e-3,set_device=True)
    # 读器件
    voltage_base = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
    voltage = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
    cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)

    cond = cond_sub_base.flatten()
    counts, bin_edges, _ = plt.hist(cond, bins=bin_edges, color='blue', alpha=0.7, edgecolor='black')
    np.save(f"result/tgv3/{int(tg*10)}v_interval{interval}_vbin_edges.npy", bin_edges)
    np.save(f"result/tgv3/{int(tg*10)}v_interval{interval}_bin_counts.npy", counts)
    max_count = np.max(counts)
    max_index = np.argmax(counts)
    print(max_index,max_count)
    # 添加标题和标签
    plt.title(f"{int(tg*10)}v_interval{interval}_{int(max_index)*interval}us_{int(max_index+1)*interval}us")
    plt.xlabel("cond(uS)")
    plt.ylabel("Frequency")

    plt.savefig(f"result/tgv3/{int(tg*10)}v_interval{interval}_{int(max_index)*interval}us_{int(max_index+1)*interval}us.png")  # 保存为 PNG 格式
    plt.show()

In [ ]:
interval = 20
bin_edges = np.linspace(0, 2000, int(2000/interval)+1)

need_read = np.ones((256,256),dtype=bool)

for i in range(16):
    tg = 0.8+0.1*i
    # 先reset
    chip.write_point2(crossbar=need_read,write_voltage=2,tg=5,pulse_width=1e-6,set_device=False)
    # 再set
    chip.write_point2(crossbar=need_read,write_voltage=3,tg=tg,pulse_width=1e-6,set_device=True)
    # 读器件
    voltage_base = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
    voltage = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
    cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)

    cond = cond_sub_base.flatten()
    counts, bin_edges, _ = plt.hist(cond, bins=bin_edges, color='blue', alpha=0.7, edgecolor='black')
    np.save(f"result/脉宽1us/{int(tg*10)}v_interval{interval}_vbin_edges.npy", bin_edges)
    np.save(f"result/脉宽1us/{int(tg*10)}v_interval{interval}_bin_counts.npy", counts)
    max_count = np.max(counts)
    max_index = np.argmax(counts)
    print(max_index,max_count)
    # 添加标题和标签
    plt.title(f"{int(tg*10)}v_interval{interval}_{int(max_index)*interval}us_{int(max_index+1)*interval}us")
    plt.xlabel("cond(uS)")
    plt.ylabel("Frequency")

    plt.savefig(f"result/脉宽1us/{int(tg*10)}v_interval{interval}_{int(max_index)*interval}us_{int(max_index+1)*interval}us.png")  # 保存为 PNG 格式
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 假设你有一个 256x256 的 NumPy 矩阵
matrix = np.random.rand(256, 256)  # 示例矩阵，你可以用你的实际矩阵替换

# 将矩阵展平为一维数组
cond = matrix.flatten()

# 自定义 bin 的边界
# 例如，将数据范围 [0, 1] 分成 10 个 bin，每个 bin 的宽度为 0.1
bin_edges = np.linspace(start=0, stop=1, num=101)  # 生成 bin 的边界数组

# 绘制直方图
plt.hist(cond, bins=bin_edges, color='blue', alpha=0.7, edgecolor='black')

# 添加标题和标签
plt.title("Histogram with Custom Bin Widths")
plt.xlabel("Value")
plt.ylabel("Frequency")

# 添加 bin 边界作为横坐标
# plt.xticks(bin_edges)  # 设置横坐标为 bin 边界

# 显示直方图
plt.show()